In [2]:
from src.acquisition import *

from pathlib import Path
root = Path.cwd()

import pandas as pd

from osgeo import gdal
gdal.SetConfigOption("GDAL_HTTP_UNSAFESSL", "YES")
gdal.SetConfigOption('GDAL_HTTP_COOKIEFILE','~/cookies.txt')
gdal.SetConfigOption('GDAL_HTTP_COOKIEJAR', '~/cookies.txt')
gdal.SetConfigOption('GDAL_DISABLE_READDIR_ON_OPEN','YES')
gdal.SetConfigOption('CPL_VSIL_CURL_ALLOWED_EXTENSIONS','TIF')

### Query API

In [ ]:
TC_FILENAME = 'tc_boston_30m.tif'
CITY = 'boston'
BOUNDARY_FILE = 'boston_boundary.gpkg'
FROM_DATE = '2018-04-01'
TO_DATE = '2018-11-30'

landsat_catalog = "HLSL30.v2.0"
landsat_bands = ["B02","B03","B04","B05","B06","B07","B10","B11","Fmask"]
sentinel_catalog = "HLSS30.v2.0"
sentinel_bands = ["B02","B03","B04","B05","B06","B07","B8A","B11","B12","Fmask"]

# read in city vector data
boundary, bbox_utm, bbox_4326, shoreline = read_vector(boundary_file=BOUNDARY_FILE,root=root,cheasapeake=False)

# query API, crop satellite data to boundaries
sentinel_crop = get_stac_items(catalog_name=sentinel_catalog,
                   from_date=FROM_DATE, 
                   to_date=TO_DATE,
                   bbox_utm=bbox_utm,
                   bbox_4326=bbox_4326,
                   boundary=boundary,
                   bands=sentinel_bands,
                   shoreline=shoreline)

FROM_DATE = '2020-04-01'
TO_DATE = '2020-11-30'
landsat_crop = get_stac_items(catalog_name=landsat_catalog,
                   from_date=FROM_DATE, 
                   to_date=TO_DATE,
                   bbox_utm=bbox_utm,
                   bbox_4326=bbox_4326,
                   boundary=boundary,
                   bands=landsat_bands,
                   shoreline=shoreline)



In [ ]:
# 2016
l_drop = check_data(landsat_crop,'first')
# manually drop times with partial data
l_drop = l_drop.isel(time=[0,3,4,5,7])

In [ ]:
# 2018
#l_sept = check_data(landsat_crop,'first')
l_sept = l_sept.isel(time=slice(2,4))
l_sept

In [102]:
# 2020
#l_nov = check_data(landsat_crop,'last')
l_nov = l_nov.isel(time=4)
l_nov = l_nov.expand_dims({'time':[l_nov.coords['time'].values]})

In [ ]:
# 2020
l_may = check_data(landsat_crop,'first')  # first and last refer to which duplicate to drop
l_may = l_may.isel(time=slice(1,3))
l_may

In [ ]:
s_drop = check_data(sentinel_crop,'first')  

# manually drop times with partial data
to_drop = s_drop.time.values[[3,5,12,15,20,21,28,31,33]]
s_drop = s_drop.drop_sel({'time':to_drop})

In [103]:
# save image dates to csv
all_landsat_times = np.concat([l_drop.time.values, l_sept.time.values,l_may.time.values,l_nov.time.values])

sdf = pd.DataFrame({'sentinel':s_drop.time.values})
ldf = pd.DataFrame({'landsat':all_landsat_times})       

new = pd.concat([sdf, ldf], axis=1) 
new.to_csv(root / 'output' / CITY / f'{CITY}_satellite_images.csv')

In [ ]:
# change landsat dates to same year and combine

# nov and may from 2020
l_nov['time'] = l_nov.time.to_index() - pd.Timedelta(days=365*4)
l_may['time'] = l_may.time.to_index() - pd.Timedelta(days=365*4)
# september from 2018
l_sept['time'] = l_sept.time.to_index() - pd.Timedelta(days=365*2)

l_combine = l_drop.combine_first(l_may).combine_first(l_sept).combine_first(l_nov)

### Cloud Mask and Scale

In [111]:
l_masked = mask_and_scale2(l_combine, sensor='landsat',scale_factor1= .0001, scale_factor2= .01)
s_masked = mask_and_scale2(s_drop,sensor='sentinel',scale_factor1= .0001, scale_factor2=None)

In [113]:
# dates need to be chanaged to the same year in order for merge to work

# sentinel from 2018
s_masked['time'] = s_masked.time.to_index() - pd.Timedelta(days=365*2)


### Calculate vegetation indices, merge landsat and sentinel data

In [115]:
all_unique_indices, all_annual_unique_indices = get_unique_indices(l_data=l_masked,s_data=s_masked)
all_shared_indices, all_annual_shared_indices = get_shared_indices(l_data=l_masked,s_data=s_masked)

t = concatenate_and_save_data(annual1=all_annual_unique_indices,annual2=all_annual_shared_indices,
                              monthly1=all_unique_indices,monthly2=all_shared_indices,root=root,filename=f'{CITY}_hls_indices2')

c:\Users\roseh\miniconda3\envs\rs-env\Lib\site-packages\dask\array\core.py:4888: PerformanceWarning: Increasing number of chunks by factor of 11
  result = blockwise(
c:\Users\roseh\miniconda3\envs\rs-env\Lib\site-packages\dask\array\core.py:4888: PerformanceWarning: Increasing number of chunks by factor of 11
  result = blockwise(


In [116]:
t

<xarray.DataArray 'stackstac-9564630fb352892a4135dadb186a6f8b' (band: 19,
                                                                time: 9,
                                                                y: 642, x: 550)> Size: 483MB
dask.array<concatenate, shape=(19, 9, 642, 550), dtype=float64, chunksize=(1, 1, 642, 550), chunktype=numpy.ndarray>
Coordinates:
  * band         (band) object 152B 'chlorophyll_index_red_edge' ... 'lswi'
  * x            (x) float64 4kB 8.141e+05 8.141e+05 ... 8.305e+05 8.306e+05
  * y            (y) float64 5kB 4.702e+06 4.702e+06 ... 4.682e+06 4.682e+06
    epsg         int64 8B 26918
    spatial_ref  int64 8B 0
  * time         (time) <U9 324B 'april' 'may' 'june' ... 'november' 'annual'

In [117]:
all_unique_bands, all_annual_unique = get_unique_bands(l_data=l_masked,s_data=s_masked)
all_shared_bands, all_annual_shared = get_shared_bands(landsat_data=l_masked,sentinel_data=s_masked)


s = concatenate_and_save_data(annual1=all_annual_shared,annual2=all_annual_unique,
                              monthly1=all_unique_bands,monthly2=all_shared_bands, root=root,filename=f'{CITY}_hls_bands2',
                              # extra arguments needed for adding tree canopy layer
                              add_canopy=True,shoreline=shoreline,tc_filename=TC_FILENAME,city_boundary=boundary)


### Save as Rasters

In [ ]:
# def save_monthly_rasters(data,filename):
#     months = ['april','may','june','july','august','september','october','november']
#     for i in range(0,8):
#         data.isel(time=i).rio.to_raster(root / 'data' / f'{filename}_{months[i]}.tif')


# all_annual.isel(time=0).rio.to_raster(root / 'data' / 'annual_bands.tif')     

# save_monthly_rasters(all_shared_bands,filename='shared_bands')
# save_monthly_rasters(all_unique_bands,filename='unique_bands')
# save_monthly_rasters(all_shared_indices,filename='shared_indices')